In [0]:
df_country2=spark.read.table("subscription_catalog.bronze_schema.br_country")
df_country2=df_country2.dropna().drop_duplicates()

In [0]:
df_customers2=spark.read.table("subscription_catalog.bronze_schema.br_customers")
df_customers2=df_customers2.dropna().drop_duplicates()
display(df_customers2)


In [0]:
from pyspark.sql.functions import col,to_date,trim
from pyspark.sql.types import IntegerType, DateType

df_customers2= df_customers2.withColumn("account_created_date",to_date(col("account_created_date"), "dd-MM-yyyy"))
print(df_customers2.dtypes)
df_customers2.head()

In [0]:
from pyspark.sql.functions import trim, col

def trim_all_string_columns(df):
    for c, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(c, trim(col(c)))
    return df

In [0]:
df_customers2 = trim_all_string_columns(df_customers2)

df_customers2.show()

In [0]:
df_employee2=spark.read.table("subscription_catalog.bronze_schema.br_employee")

In [0]:
from pyspark.sql.functions import when, col, to_date

df_employee2 = df_employee2.withColumn("hire_date", to_date(col("hire_date"), "dd-MM-yyyy")) \
       .withColumn("last_update", to_date(col("last_update"), "dd-MM-yyyy"))

df_employee2 = df_employee2.withColumn("is_active_flag",when(col("is_active_flag") == "Yes", 1).otherwise(0))
df_employee2=df_employee2.dropna().drop_duplicates()
df_employee2 = trim_all_string_columns(df_employee2)

df_employee2.show()


In [0]:
df_fxrate2=spark.read.table("subscription_catalog.bronze_schema.br_fxrate")
display(df_fxrate2)


In [0]:
df_fxrate2 = df_fxrate2.withColumn("effective_date",to_date(col("effective_date"), "yyyy-MM-dd"))
df_fxrate2 = trim_all_string_columns(df_fxrate2)
df_fxrate2=df_fxrate2.dropna().drop_duplicates()
df_fxrate2.show()

In [0]:
df_opportunity2=spark.read.table("subscription_catalog.bronze_schema.br_opportunity")
df_opportunity2 = trim_all_string_columns(df_opportunity2)
df_opportunity2=df_opportunity2.dropna().drop_duplicates()
display(df_opportunity2)

In [0]:
from pyspark.sql.functions import regexp_replace

df_opportunity2 = df_opportunity2.withColumn("start_date", to_date(col("start_date"), "dd-MM-yyyy")) \
       .withColumn("end_date", to_date(col("end_date"), "dd-MM-yyyy"))
df_opportunity2 = df_opportunity2.dropna()



In [0]:
df_products2=spark.read.table("subscription_catalog.bronze_schema.br_products")
df_products2 = df_products2.withColumn("created_date",to_date(col("created_date"), "dd-MM-yyyy"))
df_products2 = trim_all_string_columns(df_products2)
df_products2 = df_products2.withColumn("is_active",when(col("is_active") == "Y", 1).otherwise(0))
df_products2=df_products2.dropna().drop_duplicates()
display(df_products2) 



In [0]:
from pyspark.sql.functions import regexp_replace, col, when, round as spark_round, coalesce, lit, to_date

df_opp_fresh = spark.read.table("subscription_catalog.bronze_schema.br_opportunity")

df_opp_fresh = df_opp_fresh.withColumn("start_date", to_date(col("start_date"), "dd-MM-yyyy")) \
       .withColumn("end_date", to_date(col("end_date"), "dd-MM-yyyy"))
df_opp_fresh = df_opp_fresh.dropna()
df_opp_fresh = trim_all_string_columns(df_opp_fresh)

display(df_opp_fresh)


In [0]:
df_with_currency = df_opp_fresh \
    .withColumn("detected_currency",
        when(col("revenue_amount").contains("£"), "GBP")
        .when(col("revenue_amount").contains("€"), "EUR")
        .when(col("revenue_amount").contains("$"), "USD")
        .otherwise(None)
    ) \
    .withColumn("revenue_numeric",
        regexp_replace(col("revenue_amount"), "[£€$,]", "").cast("double")
    )
display(df_with_currency.limit(100))


In [0]:
customers_slim = df_customers2.select(
    col("customer_id").alias("c_customer_id"),
    col("country").alias("customer_country")
)

df_with_country = df_with_currency.join(
    customers_slim,
    df_with_currency["customer_id"] == customers_slim["c_customer_id"],
    "left"
).drop("c_customer_id")

display(df_with_country.limit(100))


In [0]:
country_slim = df_country2.select(
    col("country_code").alias("c_country_code"),
    col("currency_code").alias("country_based_currency")
)

df_with_country_currency = df_with_country.join(
    country_slim,
    df_with_country["customer_country"] == country_slim["c_country_code"],
    "left"
).drop("c_country_code")

display(df_with_country_currency .limit(100))


In [0]:
df_with_final_currency = df_with_country_currency.withColumn(
    "transaction_currency",
    coalesce(col("detected_currency"), col("country_based_currency"))
)
display(df_with_final_currency.limit(100))

fx_slim = df_fxrate2.select(
    col("currency_code").alias("fx_currency_code"),
    col("fx_rate_to_gbp")
)
display(fx_slim.limit(100))

df_with_fx_rate = df_with_final_currency.join(
    fx_slim,
    df_with_final_currency["transaction_currency"] == fx_slim["fx_currency_code"],
    "left"
).drop("fx_currency_code")
display(df_with_fx_rate.limit(100))



In [0]:
df_opportunity_in_gbp = df_with_fx_rate.withColumn(
    "revenue_in_gbp",
    spark_round(col("revenue_numeric") * coalesce(col("fx_rate_to_gbp"), lit(1.0)), 2)
)

df_opportunity_silver = df_opportunity_in_gbp.select(
    "opportunity_id",
    "customer_id",
    "product_id",
    "employee_id",
    "start_date",
    "end_date",
    "contract_term",
    "revenue_amount",
    "customer_country",
    "transaction_currency",
    "fx_rate_to_gbp",
    "revenue_numeric",
    "revenue_in_gbp",
    "close_status",
    "created_timestamp"
)

# only keep list price from product table
df_opportunity_silver = df_opportunity_silver.join(
    df_products2.select("product_id", "list_price"),
    "product_id",
    "left"
)

print(f"Total opportunities with GBP conversion: {df_opportunity_silver.count()}")
display(df_opportunity_silver.limit(50))

In [0]:
df_country2.write.format("delta").mode("overwrite").saveAsTable("subscription_catalog.silver_schema.sl_country")
df_customers2.write.format("delta").mode("overwrite").saveAsTable("subscription_catalog.silver_schema.sl_customers")
df_employee2.write.format("delta").mode("overwrite").saveAsTable("subscription_catalog.silver_schema.sl_employee")
df_fxrate2.write.format("delta").mode("overwrite").saveAsTable("subscription_catalog.silver_schema.sl_fxrate")
df_opportunity2.write.format("delta").mode("overwrite").saveAsTable("subscription_catalog.silver_schema.sl_opportunity")
df_products2.write.format("delta").mode("overwrite").saveAsTable("subscription_catalog.silver_schema.sl_products")
df_opportunity_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("subscription_catalog.silver_schema.sl_opportunity_main")